[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S04/S04_04_ficheros_excepciones.ipynb)

# S04_04 · Ficheros CSV y JSON con la biblioteca estándar, y excepciones
**Sprint 4 · Python para IA · ⏱ 60 min**

## Qué vas a aprender
1. A **leer un CSV** con el módulo `csv` y a convertir los textos a números.
2. A **guardar y leer JSON** con el módulo `json`.
3. A manejar errores con `try` / `except` / `finally` y a lanzar los tuyos con `raise`.
4. A construir un pequeño lector **robusto** de datos: la base del reto del sprint.

## Contexto TurisData
El cliente nos envía sus reservas en un CSV (`reservas_turisdata.csv`). Los ficheros reales fallan: faltan
columnas, hay celdas vacías, alguien escribe "tres" en vez de 3. Un buen script **no se cae**: informa.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import csv
import json
import os
import tempfile

ruta_reservas = dato("reservas_turisdata.csv")
print(ruta_reservas)

## 1 · Leer un CSV con `csv.DictReader`
`DictReader` devuelve cada fila como un **diccionario** (la cabecera da las claves). **Todos los valores son
texto**: hay que convertirlos con `int()` y `float()`. Abrimos con `with` para que el archivo se cierre solo,
y con `newline=""` y `encoding="utf8"` para evitar problemas con saltos de línea y tildes.

In [ ]:
with open(ruta_reservas, newline="", encoding="utf8") as f:
    filas = list(csv.DictReader(f))

print(len(filas), "filas")
print(filas[0])
print(type(filas[0]["noches"]), "-> hay que convertir con int()")

### Ejercicio 1 · Contar
Con la lista `filas`, guarda en `n_filas` cuántas reservas hay y en `n_canceladas` cuántas tienen
`cancelada` igual a `"1"` (recuerda: llega como texto; puedes convertirla con `int`).

In [ ]:
# TODO: len(filas) y un sum sobre int(fila["cancelada"]) (o una comprensión con filtro)
...
assert n_filas == 6000, f"El fichero tiene 6000 reservas y has contado {n_filas}"
assert n_canceladas == 2754, f"Deberían ser 2754 canceladas y has contado {n_canceladas}"
print("Correcto:", n_filas, n_canceladas)

### Ejercicio 2 · Ingreso medio de las reservas confirmadas
Calcula `ingreso_medio` = media de `noches * precio_noche` **solo de las reservas no canceladas**,
redondeada a 2 decimales. Recuerda convertir `noches` a `int` y `precio_noche` a `float`.

In [ ]:
# TODO: filtra las filas con cancelada == "0", calcula noches*precio de cada una y divide la suma entre cuántas son
...
assert ingreso_medio == 576.04, f"El ingreso medio por reserva confirmada es 576.04 y tienes {ingreso_medio}"
print("Correcto:", ingreso_medio)

## 2 · Escribir y leer JSON
JSON es el formato de intercambio de las APIs. `json.dump` escribe un diccionario en un archivo y
`json.load` lo lee. Con `ensure_ascii=False` se conservan las tildes y con `indent=2` queda legible.
Escribimos en una carpeta temporal para no dejar archivos por ahí.

In [ ]:
carpeta = tempfile.mkdtemp()
ruta_json = os.path.join(carpeta, "resumen.json")

resumen = {"cliente": "TurisData Canarias", "reservas": 6000, "canales": ["directo", "agencia"]}
with open(ruta_json, "w", encoding="utf8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)

print(open(ruta_json, encoding="utf8").read())
with open(ruta_json, encoding="utf8") as f:
    leido = json.load(f)
print(leido == resumen)

### Ejercicio 3 · Guardar KPIs en JSON
Crea `ruta_kpis` (dentro de `carpeta`, con nombre `"kpis.json"`) y guarda en él el diccionario
`{"n_filas": n_filas, "n_canceladas": n_canceladas}` con los valores del ejercicio 1. Después léelo de nuevo y
guarda el resultado en `kpis_leidos`.

In [ ]:
# TODO: os.path.join(carpeta, "kpis.json"); json.dump al escribir (modo "w") y json.load al leer
...
assert os.path.basename(ruta_kpis) == "kpis.json", "El archivo debe llamarse kpis.json"
assert kpis_leidos == {"n_filas": 6000, "n_canceladas": 2754}, f"Lo leído no coincide con lo esperado: {kpis_leidos}"
print("Correcto:", kpis_leidos)

## 3 · Excepciones: cuando algo falla
Un error en Python es una **excepción**. Si no se captura, el programa se detiene. Con `try/except` decides
qué hacer. Excepciones frecuentes: `FileNotFoundError`, `ValueError` (conversión imposible), `KeyError`
(clave inexistente), `ZeroDivisionError`.
- `except TipoError:` captura **ese** tipo (nunca uses un `except:` vacío: esconde fallos reales).
- `else:` se ejecuta si no hubo error; `finally:` se ejecuta siempre (útil para limpiar).
- `raise` lanza una excepción propia.

In [ ]:
for texto in ["89.24", "", "tres"]:
    try:
        precio = float(texto)
    except ValueError:
        print(f"{texto!r}: no es un número válido")
    else:
        print(f"{texto!r}: {precio}")
    finally:
        pass   # aquí iría la limpieza (cerrar conexiones, etc.)

try:
    open("no_existe.csv", encoding="utf8")
except FileNotFoundError as error:
    print("Aviso:", error)

### Ejercicio 4 · Conversión segura
Escribe `a_float_seguro(texto, defecto=None)` que devuelva `float(texto)` y, si el texto no se puede
convertir (`ValueError`) o es `None` (`TypeError`), devuelva `defecto`.

In [ ]:
# TODO: try: return float(texto) / except (ValueError, TypeError): return defecto
...
assert a_float_seguro("89.24") == 89.24, "Un número válido se convierte"
assert a_float_seguro("tres") is None, "Un texto no numérico devuelve el valor por defecto (None)"
assert a_float_seguro("", defecto=0.0) == 0.0, "Una cadena vacía devuelve el valor por defecto que se pase"
assert a_float_seguro(None, defecto=-1.0) == -1.0, "None también debe manejarse (TypeError)"
print("Correcto")

### Ejercicio 5 · Lector de CSV con avisos claros
Escribe `leer_csv(ruta, columnas_obligatorias)` que:
1. Abra el CSV y devuelva la lista de filas (diccionarios).
2. Si el archivo **no existe**, lance `FileNotFoundError` con un mensaje claro que incluya la ruta.
3. Si **falta alguna columna obligatoria** en la cabecera, lance `ValueError` con un mensaje que nombre las columnas que faltan.

Pista: `csv.DictReader(f).fieldnames` da la cabecera.

In [ ]:
# TODO: captura FileNotFoundError y vuelve a lanzarlo con raise ... from error; calcula las columnas que faltan con un conjunto
...
assert len(leer_csv(ruta_reservas, ["canal", "noches"])) == 6000, "Con un CSV correcto debe devolver las 6000 filas"
try:
    leer_csv(os.path.join(carpeta, "no_existe.csv"), ["canal"])
    raise AssertionError("Debería lanzar FileNotFoundError si el fichero no existe")
except FileNotFoundError as error:
    assert "no_existe.csv" in str(error), "El mensaje debe incluir la ruta del fichero"
try:
    leer_csv(ruta_reservas, ["canal", "columna_inventada"])
    raise AssertionError("Debería lanzar ValueError si falta una columna")
except ValueError as error:
    assert "columna_inventada" in str(error), "El mensaje debe nombrar la columna que falta"
print("Correcto")

## 4 · Mini-reto integrador: resumen tolerante a filas malas
Un cliente nos manda un CSV con **filas defectuosas**. Simulamos uno pequeño (más abajo) con
`noches` no numérico o vacío. Escribe `resumen_canales(ruta)` que:
- lea el fichero con tu `leer_csv` (columnas obligatorias `canal`, `noches`, `precio_noche`);
- **descarte** (sin romper el programa) las filas cuyo `noches` o `precio_noche` no se puedan convertir a número;
- devuelva un diccionario `{"reservas_validas": int, "filas_descartadas": int, "noches_por_canal": {canal: noches totales}}`.

In [ ]:
ruta_mini = os.path.join(carpeta, "mini.csv")
with open(ruta_mini, "w", newline="", encoding="utf8") as f:
    f.write("canal,noches,precio_noche\n"
            "directo,3,80.0\n"
            "agencia,tres,60.0\n"        # noches no numérico
            "directo,4,\n"               # precio vacío
            "web_intermediaria,5,95.5\n"
            "agencia,2,150.0\n")

# TODO: recorre las filas con leer_csv; convierte con int/float dentro de try/except ValueError y acumula
...
resultado = resumen_canales(ruta_mini)
assert resultado["reservas_validas"] == 3, f"Hay 3 filas válidas y tienes {resultado['reservas_validas']}"
assert resultado["filas_descartadas"] == 2, f"Se descartan 2 filas defectuosas y tienes {resultado['filas_descartadas']}"
assert resultado["noches_por_canal"] == {"directo": 3, "web_intermediaria": 5, "agencia": 2}, f"Noches por canal incorrectas: {resultado['noches_por_canal']}"
print("Correcto:", json.dumps(resultado, ensure_ascii=False))

## 5 · Para reflexionar
1. ¿Por qué es peligroso un `except:` vacío? ¿Y descartar filas sin contarlas?
2. Si el cliente cambia el nombre de una columna sin avisar, ¿qué prefieres: que tu script falle con un mensaje claro o que calcule datos erróneos?
3. ¿Qué diferencia hay entre `json.dump` y `json.dumps`?

## 6 · Qué has aprendido
- Leer CSV con `csv.DictReader` (todo llega como texto) y guardar/leer JSON.
- `try` / `except` / `else` / `finally` y `raise`; excepciones concretas frente a `except:` genérico.
- Un lector **robusto**: valida la cabecera, cuenta lo que descarta y avisa con mensajes claros.
- Esto es justo lo que necesitarás en el reto **«Herramientas internas»**.